In [1]:
import json
with open("r1_samples.json", "r", encoding="utf-8") as f:
    r1_samples = json.load(f)

In [2]:
print(len(r1_samples))

4782


In [3]:
r1_samples[0]

{'q*': 'When selecting a current sensing resistor for use in automotive applications requiring high reliability under extreme thermal and mechanical stress, which performance characteristic should be prioritized to ensure minimal resistance drift during prolonged exposure to high temperatures?',
 'a*': ['E'],
 'options*': ['A resistance',
  'B partial discharge',
  'C coast down time',
  'D vibration',
  'E temperature'],
 'documents': ['Current Sensing Resistors, Metal Plate Type\n\n## Performance (AEC-Q200)\n\n### ● ERJMS4S/ERJMS4H\n\n<table><thead><tr><th>Test item</th><th>Performance requirements ΔR</th><th>Typical value ΔR</th><th>Test condition</th></tr></thead><tbody><tr><td>Thermal shock</td><td>±1 %</td><td>0.20 %</td><td>-55 °C / +155 °C, 1000 cycles</td></tr><tr><td>Overload</td><td>±0.5 %</td><td>0.10 %</td><td>Rated power x 3, 5 s</td></tr><tr><td>Solderability</td><td>> 95% coverage</td><td>> 95% coverage</td><td>245 °C, 3 s</td></tr><tr><td>Resistance to solvents</td><td

In [4]:
import re
import string

def has_full_lettered_options(sample, min_options=2):
    """
    Detect if options are in the options* field as full options.
    Returns True if options* contains entries that look like complete options
    (with identifier and text).
    
    Handles various formats:
    - Standard lettered: "A. text", "B) text", "C: text"
    - Numbered: "1. text", "2) text"
    - Mixed/special: "#2 combination relay..."
    - Simple identifiers: "A text" (without punctuation)
    """
    options = sample.get("options*", [])
    
    if not options or len(options) < min_options:
        return False
    
    # Count how many options look like "full" options (identifier + text)
    full_option_count = 0
    all_single_identifiers = True
    
    for opt in options:
        if not isinstance(opt, str):
            # If it's not a string, it's not a "full" option
            all_single_identifiers = False
            continue
            
        opt = opt.strip()
        
        # Check if it's just a single identifier (like "A", "B", "1", "#2")
        if len(opt) <= 5 and re.match(r'^[A-Z0-9#]+$', opt):
            # It's just an identifier, not a full option
            continue
        
        # Check for patterns that indicate a full option:
        # 1. Starts with identifier followed by punctuation and text
        #    e.g., "A. text", "1) text", "#2: text"
        pattern1 = re.compile(r'^[A-Z0-9#]+[\.\)\:]?\s+.+')
        
        # 2. Starts with identifier followed by space and text (no punctuation)
        #    e.g., "A oil charge", "1 relay model"
        pattern2 = re.compile(r'^([A-Z0-9#]+)\s+[A-Za-z].+')
        
        # 3. Contains multiple words/significant text (not just an identifier)
        words = opt.split()
        if len(words) >= 2 and any(word[0].isalpha() for word in words[1:] if word):
            # Has at least 2 words and the second word starts with a letter
            full_option_count += 1
        elif pattern1.match(opt) or pattern2.match(opt):
            full_option_count += 1
    
    # If majority of options look like full options, return True
    return full_option_count >= min_options and full_option_count >= len(options) * 0.6


In [5]:
import re
import string

def has_embedded_options_in_q(sample, min_options=2):
    """
    Detect if options are embedded in the question string.
    Returns True if the question contains lettered options (A, B, C, etc.)
    with at least min_options distinct options.
    """
    q = sample.get("q*", "")
    if not q:
        return False
    
    # ---------- 1️⃣ MULTILINE OPTIONS ----------
    # Check for options on separate lines: A. ...\nB. ...\nC. ...
    multiline_pattern = re.compile(r'(?m)^\s*([A-Z])[\.\)\:]?\s+.+$')
    multiline_matches = multiline_pattern.findall(q)
    if len(set(multiline_matches)) >= min_options:
        # Check if they're contiguous (A, B, C, ...)
        if len(multiline_matches) >= min_options:
            letters = sorted(set(multiline_matches))
            expected = list(string.ascii_uppercase[:len(letters)])
            if letters == expected:
                return True
    
    # ---------- 2️⃣ INLINE OPTIONS WITH "Options:" MARKER ----------
    # Check for patterns like: Options: A. ... B. ... C. ...
    if "Options:" in q or "options:" in q:
        # Find the text after "Options:" (case-insensitive)
        options_match = re.search(r'(?:[Oo]ptions:|\n)\s*(.+?)(?:\n\n|\n[A-Z]\.|\n\d\.|$)', 
                                 q, re.DOTALL | re.IGNORECASE)
        
        if options_match:
            options_text = options_match.group(1)
            
            # Pattern to match lettered options
            # This handles: A. text, A) text, A: text, A text (with spaces)
            pattern = re.compile(r'([A-Z])[\.\)\:]?\s+(.+?)(?=\s+[A-Z][\.\)\:]?\s+|\s*$|\.\s*$)', 
                               re.DOTALL)
            
            matches = []
            pos = 0
            while pos < len(options_text):
                match = pattern.match(options_text[pos:])
                if match:
                    matches.append(match.group(1))  # Just the letter
                    pos += match.end()
                else:
                    # Try to skip non-matching text
                    pos += 1
            
            if len(set(matches)) >= min_options:
                # Check if they start from A and are contiguous
                letters = sorted(set(matches))
                expected = list(string.ascii_uppercase[:len(letters)])
                if letters == expected:
                    return True
    
    # ---------- 3️⃣ DIRECT PATTERN MATCH IN ENTIRE QUESTION ----------
    # Look for patterns like: A) ... B) ... C) ... anywhere in the question
    # This catches cases without explicit "Options:" marker
    
    # Pattern for lettered options with common separators
    letter_patterns = [
        r'([A-Z])[\.\)\:]?\s+[^A-Z]*?(?=\s+[A-Z][\.\)\:]?\s+|\s*$)',
        r'\b([A-Z])[\.\)\:]\s+',
        r'\(\s*([A-Z])\s*\)\s+'
    ]
    
    all_matches = []
    for pattern_str in letter_patterns:
        matches = re.findall(pattern_str, q, re.IGNORECASE)
        all_matches.extend(matches)
    
    if len(set(all_matches)) >= min_options:
        letters = sorted(set(all_matches))
        # Check if they form a reasonable sequence
        if len(letters) >= min_options:
            # Allow some flexibility - they should be mostly contiguous
            expected = list(string.ascii_uppercase[:max(5, len(letters))])
            found_in_expected = [l for l in letters if l in expected[:len(letters)+2]]
            if len(found_in_expected) >= min_options:
                return True
    
    # ---------- 4️⃣ CHECK IF options* FIELD IS JUST LETTERS ----------
    # If options* field contains just letters (A, B, C, etc.), 
    # then the actual text is likely embedded in the question
    options_list = sample.get("options*", [])
    if options_list and len(options_list) >= min_options:
        # Check if all options are single letters
        all_single_letters = all(len(str(opt).strip()) == 1 and 
                                str(opt).strip().isalpha() and 
                                str(opt).strip().isupper() 
                                for opt in options_list)
        if all_single_letters:
            # Check if question contains detailed text for these options
            # by looking for patterns that might contain the option text
            letters_in_q = re.findall(r'([A-Z])[\.\)\:]?\s+[^A-Z]{10,}', q)
            if len(set(letters_in_q)) >= min_options:
                return True
    
    return False

In [6]:
import re

def embed_options_into_question(sample):
    """
    Embed options from options* field into the q* field.
    Returns a new sample with embedded options.
    """
    if not sample.get('options*'):
        return sample
    
    q = sample.get('q*', '')
    options = sample.get('options*', [])
    
    # Check if options are already embedded in the question
    # Look for patterns like "Options:" followed by option text
    if re.search(r'[Oo]ptions:\s*([A-Z0-9#])', q):
        # Options already seem to be embedded
        return sample
    
    # Create embedded options text
    embedded_options = "Options: "
    for i, opt in enumerate(options):
        if isinstance(opt, str):
            # Format based on the option content
            opt = opt.strip()
            
            # If option starts with letter and space (like "A speed")
            if re.match(r'^([A-Z])\s+', opt):
                # Keep as is: "A speed and position sensors"
                embedded_options += opt
            # If option starts with letter and punctuation (like "A. speed")
            elif re.match(r'^([A-Z])[\.\)\:]', opt):
                # Keep as is
                embedded_options += opt
            # If option starts with just a letter (like "A")
            elif re.match(r'^[A-Z]$', opt):
                # This shouldn't happen if we're embedding, but handle it
                embedded_options += opt
            # If option doesn't start with identifier, add one
            else:
                # Add letter identifier
                identifier = chr(ord('A') + i) if i < 26 else str(i)
                embedded_options += f"{identifier}. {opt}"
            
            # Add separator (space) between options
            if i < len(options) - 1:
                embedded_options += " "
    
    # Add embedded options to the question
    new_q = q.rstrip()
    if not new_q.endswith(('?', '.', '!')):
        new_q += '.'
    new_q += " " + embedded_options
    
    # Create new sample with embedded options
    new_sample = sample.copy()
    new_sample['q*'] = new_q
    
    return new_sample



In [7]:
r1_samples_filter1 = [item for item in r1_samples if has_full_lettered_options(item)]
print(len(r1_samples_filter1))

2526


In [8]:
r1_samples_filter2 = [item for item in r1_samples if has_embedded_options_in_q(item)]
print(len(r1_samples_filter2))

302


In [9]:
remain_r1 = [
    item for item in r1_samples
    if item not in r1_samples_filter1
    and item not in r1_samples_filter2
]
print(len(remain_r1))

1961


In [10]:
r1_samples_filter1=[embed_options_into_question(sample) for sample in r1_samples_filter1]

In [11]:
merged_list = [*r1_samples_filter1, *r1_samples_filter2]
print(f"Final r1 sample after filter {len(merged_list)}")

Final r1 sample after filter 2828


In [12]:
with open('r1_samples_filter.json', 'w', encoding='utf-8') as f:
    json.dump(merged_list, f, ensure_ascii=False, indent=2)